# 02 — Rapidly-Exploring Random Tree (RRT)

**Section:** Motion Planning · **Mirrors MATLAB:** *RRT Planners for Mobile Robots*

RRT incrementally builds a tree by sampling random points in the configuration space, finding the nearest tree node, and growing a small step toward the sample.


## Intuition — what's actually going on?

Suppose you have to plan a path through a cluttered space, and the space is *high-dimensional* (think of a robot arm with 7 joints — that's a 7-D space, way too big for a grid). A\* would blow up. RRT (Rapidly-exploring Random Tree) is the clever workaround.

The trick is **biased random exploration**: throw random darts at the configuration space, then for each dart, find the closest point on your existing tree and grow a small branch toward it. Because you're sampling uniformly, the tree naturally pushes outward into unexplored regions — like roots spreading through soil.

Two practical twists:

- **Goal bias**: every now and then (say 10% of the time), aim the dart *at the goal* instead of randomly. Without this, RRT would explore forever without ever bothering to actually reach the goal.
- **Collision check**: only add a branch if the line from the nearest tree node to the new dart is collision-free — *including* the final connecting edge to the goal.

RRT finds *a* path with probability 1 as time → ∞, but not necessarily the *shortest* one (that needs RRT*, which rewires the tree as it grows). RRT itself avoids the $2^d$ grid blow-up that kills A*, so finding *some* feasible path is much faster in high $d$. But the convergence to the *optimal* path (RRT*) still slows as $(\log n/n)^{1/d}$ — the curse of dimensionality reappears in the optimality rate.


## Analytical derivation

**Algorithm.** Let $\mathcal{X} \subset \mathbb{R}^d$ be the configuration space and $\mathcal{X}_\text{free} \subseteq \mathcal{X}$ the collision-free subset. We grow a tree $\mathcal{T} = (V, E)$ rooted at $x_\text{init}$:

1. Sample $x_\text{rand} \in \mathcal{X}$ (uniform), or with probability $p_g$ set $x_\text{rand} \leftarrow x_\text{goal}$ (goal bias).
2. Find nearest neighbor in tree: $x_\text{near} = \arg\min_{x \in V} \|x - x_\text{rand}\|$.
3. Steer: $x_\text{new} = x_\text{near} + \min(\eta,\ \|x_\text{rand} - x_\text{near}\|) \cdot \frac{x_\text{rand} - x_\text{near}}{\|x_\text{rand} - x_\text{near}\|}$, where $\eta$ is the step size (matches the code's `min(η, dist)` form).
4. If both $x_\text{new}$ and the line segment $(x_\text{near}, x_\text{new})$ are collision-free, add $x_\text{new}$ to $V$ and edge $(x_\text{near}, x_\text{new})$ to $E$.
5. Stop when $\|x_\text{new} - x_\text{goal}\| < \tau$ (goal tolerance).

**Probabilistic completeness theorem (LaValle, 1998).** Let $(X_n)_{n\ge 1}$ be iid uniform on $\mathcal{X}$ under the canonical product measure $\mathbb{P}$ on $\mathcal{X}^\infty$, and let $A_n$ be the event "RRT, applied to the first $n$ samples, returns a feasible path." If $\mathcal{X}_\text{free}$ has non-empty interior and a feasible path exists with **positive clearance** $\delta > 0$ (i.e., $\inf_{x \in \pi^*, y \notin \mathcal{X}_\text{free}} \|x - y\| \ge \delta$), and the step size satisfies $\eta \le \delta/2$, then

$$\lim_{n \to \infty} \Pr[A_n] = 1.$$

Because $A_n \subseteq A_{n+1}$ (success is monotone in samples), by continuity of measure this is equivalent to $\Pr[\text{eventual success}] = 1$ (almost-sure success).

**Not asymptotically optimal.** Unlike RRT*, plain RRT does *not* converge to the optimal-cost path (Karaman & Frazzoli 2011). RRT* additionally rewires neighbors within radius $r_n = \gamma (\log n / n)^{1/d}$ on each insertion and provably converges to the optimal path as $n \to \infty$, with $\gamma > 2(1 + 1/d)^{1/d}(\mu(\mathcal{X}_\text{free}) / \zeta_d)^{1/d}$ where $\zeta_d$ is the Lebesgue measure of the unit ball in $\mathbb{R}^d$ (Karaman-Frazzoli 2011, Thm 38). The $1/d$ exponent is the metric-entropy scaling of uniform measure on $[0,1]^d$ — the curse of dimensionality reappears in the optimality rate.

**Goal bias.** With $p_g = 0.1$ each sample is the goal with probability $0.1$. This trades off exploration vs. exploitation; $p_g \to 1$ degrades to a greedy planner that gets stuck behind obstacles.

### Compatibility check — math ↔ code

| Step | Code |
|---|---|
| Goal bias sample | `rnd = goal if np.random.random() < 0.1 else np.random.uniform(0, world, 2)` |
| Nearest neighbor (brute force) | `i_n = int(np.argmin([np.linalg.norm(p - rnd) for p in tree]))` |
| Steer by fixed step $\eta$ | `new = near + (dirn / dist) * step if dist > step else rnd` |
| Free-space + edge collision check | `if in_col(new) or edge_col(near, new): continue` |
| Add node + parent edge | `tree.append(new); parent[len(tree)-1] = i_n` |
| Goal tolerance $\tau$ | `if np.linalg.norm(new - goal) < step: ... break` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

world = 100.0
obstacles = [(30, 30, 15), (60, 70, 12), (75, 30, 10), (20, 70, 8), (50, 45, 8)]
start = np.array([5.0, 5.0])
goal = np.array([95.0, 95.0])


In [ ]:
def in_collision(pt, obs):
    for cx, cy, r in obs:
        if (pt[0] - cx) ** 2 + (pt[1] - cy) ** 2 < r * r:
            return True
    return False


def edge_collision(p1, p2, obs, steps=25):
    for t in np.linspace(0, 1, steps):
        if in_collision(p1 * (1 - t) + p2 * t, obs):
            return True
    return False


def rrt(start, goal, obs, world=100.0, max_iter=3000, step=4.0, goal_bias=0.1, goal_tol=4.0):
    tree = [start]
    parent = [-1]                          # parent[i] = index of tree[i]'s parent; -1 for root
    for _ in range(max_iter):
        rnd = goal if np.random.random() < goal_bias else np.random.uniform(0, world, 2)
        dists = [np.linalg.norm(p - rnd) for p in tree]
        i_near = int(np.argmin(dists))
        near = tree[i_near]
        direction = rnd - near
        d = np.linalg.norm(direction)
        new = near + (direction / d) * step if d > step else rnd
        if in_collision(new, obs) or edge_collision(near, new, obs):
            continue
        tree.append(new)
        parent.append(i_near)
        # COUNCIL FIX (pass 2 BLOCKER, Frazzoli): goal-connection edge MUST be
        # collision-checked, otherwise the planner can output an edge through
        # an obstacle adjacent to the goal.
        if np.linalg.norm(new - goal) < goal_tol and not edge_collision(new, goal, obs):
            tree.append(goal)
            parent.append(len(tree) - 2)
            return tree, parent, True
    return tree, parent, False


tree, parent, success = rrt(start, goal, obstacles)
print(f"Success: {success}  ·  tree size: {len(tree)} nodes")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for cx, cy, r in obstacles:
    ax.add_patch(plt.Circle((cx, cy), r, color='lightgrey'))
for idx, p_idx in enumerate(parent):
    if p_idx >= 0:
        ax.plot([tree[idx][0], tree[p_idx][0]], [tree[idx][1], tree[p_idx][1]],
                'g-', lw=0.5, alpha=0.6)
if success:
    path, idx = [], len(tree) - 1
    while idx != -1:
        path.append(tree[idx])
        idx = parent[idx]
    path = np.array(path[::-1])
    ax.plot(path[:, 0], path[:, 1], 'r-', lw=2.5, label='Path')
ax.plot(*start, 'go', markersize=14, label='Start')
ax.plot(*goal, 'b*', markersize=18, label='Goal')
ax.set_xlim(0, world); ax.set_ylim(0, world); ax.set_aspect('equal')
ax.legend(loc='upper left')
ax.set_title(f'RRT — {len(tree)} nodes, goal-bias 10%')
plt.tight_layout()
plt.show()


## Notes

- **RRT\*** rewires the tree on each insertion to converge toward an optimal path.
- **BiRRT** grows two trees (one from start, one from goal) and tries to connect them — usually faster in cluttered spaces.
- The goal-bias parameter trades exploration vs exploitation.


## References & rigor notes

**Theorem** (Probabilistic completeness; LaValle, 1998).
*If $\mathcal{X}_\text{free}$ has non-empty interior and there exists a feasible path from $x_\text{init}$ to $x_\text{goal}$ with positive clearance, then*
$$\lim_{n \to \infty} \Pr[\text{RRT finds a feasible path in } n \text{ samples}] = 1.$$

*Idea of proof.* Let $\delta > 0$ be the path clearance and $\eta \le \delta/2$ the step size. Cover the feasible path $\pi^*$ by $K = O(L/\eta)$ overlapping (Euclidean) balls $B_k$ of radius $\eta$, each contained in $\mathcal{X}_\text{free}$. Each iid uniform sample has probability $p_k > 0$ of falling in $B_k$, so $\Pr[B_k \text{ not entered after } n \text{ samples}] = (1 - p_k)^n$. Summing over $n$ gives a geometric series; Borel-Cantelli (Lemma 1) yields $\Pr[\text{eventually all } K \text{ balls entered}] = 1$. Once all $K$ balls are entered, the steering rule extends the tree along the chain, connecting start to goal. Hence $\Pr[A_n] \to 1$. ∎

For configuration spaces on robot Lie groups, replace the Euclidean ball with the Riemannian distance ball — the same argument works with manifold-dependent constants. (Halton-RRT / Sobol-RRT variants — Branicky et al. 2001 — replace iid uniform with low-discrepancy sequences for better space coverage.)

**Not asymptotically optimal.** Karaman & Frazzoli (2011) proved plain RRT does *not* converge to the optimal-cost path even as $n \to \infty$. **RRT\*** fixes this by rewiring neighbors within radius $r_n = \gamma(\log n / n)^{1/d}$ at each insertion, achieving almost-sure convergence to the optimum.

**Complexity per iteration.** $O(n)$ for brute-force nearest neighbor (or $O(\log n)$ with a kd-tree); collision check $O(d)$ where $d$ is segment discretization.

**References.**
- LaValle, S. M. (1998). *Rapidly-exploring random trees: A new tool for path planning*. TR 98-11, Iowa State University.
- LaValle, S. M. (2006). *Planning Algorithms*, Cambridge University Press, ch. 5.5.
- Karaman, S., & Frazzoli, E. (2011). *Sampling-based algorithms for optimal motion planning*. IJRR 30(7).
